# 1 - Config & Import

## 1.1 - Config

In [5]:
# 🔧 Config import
import os

logger = colored_logger()
current_file = os.path.basename(__file__) if '__file__' in globals() else "Notebook"
logger.info(f"Logger initialized ({current_file})")

[2025-06-28 11:25:23] INFO - 1353130262.py - Logger initialized (Notebook)


## 1.2 - Import

In [6]:
# 📦 External libs
import plotly.graph_objects as go
import importlib

# 📂 Internal modules
from Class import Data, Models, Trainer
import Class.Data.Features
import Class.Data.Labels
import Class.Data.Preprocessing
import Class.Data.DataAnalysis


def reload_modules():
    importlib.reload(Class.Data.DataFetcher)
    importlib.reload(Class.Data.Features)
    importlib.reload(Class.Data.Labels)
    importlib.reload(Class.Data.DataAnalysis)
    importlib.reload(Class.Data.Preprocessing)
    importlib.reload(Class.Models.LSTMModel)
    importlib.reload(Class.Trainer.LSTMTrainer)

[2025-06-28 11:25:32] INFO - DataFetcher.py - Logger initialized (DataFetcher.py)
[2025-06-28 11:25:32] INFO - Features.py - Logger initialized (Features.py)
[2025-06-28 11:25:32] INFO - Labels.py - Logger initialized (Labels.py)
[2025-06-28 11:25:32] INFO - Preprocessing.py - Logger initialized (Preprocessing.py)
[2025-06-28 11:25:32] INFO - LSTMModel.py - Logger initialized (LSTMModel.py)
[2025-06-28 11:25:35] INFO - DataAnalysis.py - Logger initialized (DataAnalysis.py)
[2025-06-28 11:25:35] INFO - LSTMTrainer.py - Logger initialized (LSTMTrainer.py)


# 2 - Data

## 2.2 - Feature Data

### Variables

In [226]:
# Resample with vwap
resample_period = '5min'

# Pivot Points
pivot_left = 15
pivot_right = 15

# Volume Pivot Points
duration_min = 240
n_cross = 7
std_factor = 1.0

# Std
std_window = 10

# Mean
mean_window = 10

### Process

In [227]:
reload_modules()

# Step 0: Initialize Features Dataframe
Features_Data = Class.Data.Features.Features(fetcher.raw_data)

# Step 1: Resample with VWAP (Liquidity)
Features_Data.resample_with_vwap(resample_period)

# Step 2: Market Sessions (Market Context)
Features_Data.market_sessions()

# Step 3: Pivot Points (Support/Resistance)
Features_Data.Pivot_Points(pivot_left, pivot_right)

# Step 4: Volume Pivot Points (Volume-based S/R)
Features_Data.Volume_Pivot_Points(duration_min, n_cross, std_factor)

# Step 5: Volume Delta (Momentum)
Features_Data.add_volume_delta()

# Step 6: Cumulative Volume Delta - CVD (Momentum)
Features_Data.add_cvd()

# Step 7: Order Book Imbalance - OBI (Pressure)
Features_Data.add_obi()

# Step 8: Price Change (Price Action)
Features_Data.add_price_change()

# Step 9: Reaction Ratio (Liquidity)
Features_Data.add_reaction_ratio()

# Step 10: Rolling Standard Deviation of Price (Volatility)
Features_Data.add_rolling_std_price(std_window)

# Step 11: Rolling Mean of CVD (Smoothed Flow)
Features_Data.add_rolling_mean_cvd(mean_window)

[2025-06-15 17:24:33] INFO - DataFetcher.py - Logger initialized (DataFetcher.py)
[2025-06-15 17:24:33] INFO - Features.py - Logger initialized (Features.py)
[2025-06-15 17:24:33] INFO - Labels.py - Logger initialized (Labels.py)
[2025-06-15 17:24:33] INFO - DataAnalysis.py - Logger initialized (DataAnalysis.py)
[2025-06-15 17:24:33] INFO - Preprocessing.py - Logger initialized (Preprocessing.py)
[2025-06-15 17:24:33] INFO - LSTMModel.py - Logger initialized (LSTMModel.py)
[2025-06-15 17:24:33] INFO - LSTMTrainer.py - Logger initialized (LSTMTrainer.py)
[2025-06-15 17:24:33] INFO - Features.py - Starting resampling with period: 5min
[2025-06-15 17:24:33] INFO - Features.py - Basic OHLCV resampling done.
[2025-06-15 17:24:33] INFO - Features.py - VWAP calculation completed.
[2025-06-15 17:24:33] INFO - Features.py - Resampling successful. Resulting shape: (293, 8)
[2025-06-15 17:24:33] INFO - Features.py - Starting market session flag generation.
[2025-06-15 17:24:33] INFO - Features.py

### Visualization

In [228]:
#-------------------------------------------------------------------------------

print("#----------------------------------------------------------------------")
print("Shape:", Features_Data.data.shape)
print("#----------------------------------------------------------------------")
print(Features_Data.data.columns)
print("#----------------------------------------------------------------------")
print(Features_Data.data.head())
print("#----------------------------------------------------------------------")

#-------------------------------------------------------------------------------
df = Features_Data.data.copy()[-300:]

# 1. Candlestick chart with VWAP_5m
fig1 = go.Figure()
fig1.add_trace(go.Candlestick(x=df.index,
                              open=df['Open'],
                              high=df['High'],
                              low=df['Low'],
                              close=df['Close'],
                              name="Candlestick"))
fig1.add_trace(go.Scatter(x=df.index, y=df['VWAP_5m'], mode='lines', name='VWAP_5m', line=dict(color='white')))
fig1.update_layout(title="Candlestick Chart with VWAP_5m", xaxis_title="Time", yaxis_title="Price", template="plotly_dark",xaxis_rangeslider_visible=False,)

# 2. Market session open status
fig2 = go.Figure()
fig2.add_trace(go.Scatter(x=df.index, y=df["London_Open"], mode='lines', name="London"))
fig2.add_trace(go.Scatter(x=df.index, y=df["NY_Open"], mode='lines', name="New York"))
fig2.add_trace(go.Scatter(x=df.index, y=df["HK_Open"], mode='lines', name="Hong Kong"))
fig2.update_layout(title="Market Sessions", xaxis_title="Time", yaxis_title="Market Open (1=open)", template="plotly_dark")

# 3. Pivot points for price
fig3 = go.Figure()
fig3.add_trace(go.Scatter(x=df.index, y=df["Low"], mode='lines', name="Low", line=dict(width=0.8)))
fig3.add_trace(go.Scatter(x=df.index, y=df["Low_Pivot"], mode='lines', name="Low Pivot", line=dict(width=1.2, dash='dash')))
fig3.add_trace(go.Scatter(x=df.index, y=df["High"], mode='lines', name="High", line=dict(width=0.8)))
fig3.add_trace(go.Scatter(x=df.index, y=df["High_Pivot"], mode='lines', name="High Pivot", line=dict(width=1.2, dash='dash')))
fig3.update_layout(title="Price Pivot Points Detection", xaxis_title="Time", yaxis_title="Price", template="plotly_dark")

# 4. Volume pivot points
fig4 = go.Figure()
fig4.add_trace(go.Scatter(x=df.index, y=df["Volume_Low_Pivot"], mode='lines', name="Volume Low Pivot", line=dict(width=1.2, dash='dash')))
fig4.add_trace(go.Scatter(x=df.index, y=df["VWAP_5m"], mode='lines', name="VWAP 5m", line=dict(width=0.8)))
fig4.add_trace(go.Scatter(x=df.index, y=df["Rolling_VWAP_240min"], mode='lines', name="Rolling VWAP 240min"))
fig4.add_trace(go.Scatter(x=df.index, y=df["Volume_High_Pivot"], mode='lines', name="Volume High Pivot", line=dict(width=1.2, dash='dash')))
fig4.update_layout(title="Volume Pivot Points Detection", xaxis_title="Time", yaxis_title="Price", template="plotly_dark")

fig1.show()
fig2.show()
fig3.show()
fig4.show()

#----------------------------------------------------------------------
Shape: (293, 27)
#----------------------------------------------------------------------
Index(['Open', 'High', 'Low', 'Close', 'Volume', 'bidSize', 'askSize',
       'VWAP_5m', 'London_Open', 'NY_Open', 'HK_Open', 'Dif_Low_Pivot',
       'Dif_High_Pivot', 'Low_Pivot', 'High_Pivot', 'Rolling_VWAP_240min',
       'Volume_Low_Pivot', 'Volume_High_Pivot', 'Dif_Volume_Low_Pivot',
       'Dif_Volume_High_Pivot', 'volume_delta', 'CVD', 'obi', 'price_change',
       'reaction_ratio', 'rolling_std_price', 'rolling_mean_cvd'],
      dtype='object')
#----------------------------------------------------------------------
                         Open      High       Low     Close      Volume  \
time                                                                      
2025-06-11 21:55:00  1.148820  1.148820  1.148820  1.148820      2000.0   
2025-06-11 22:00:00  1.148830  1.149390  1.148820  1.149355   3265700.0   
2025-06-11

## 2.3 - Label Data

### Variables

In [229]:
# Step 1: Categorize Pivot Points
look_forward=20

# Step 2: Categorize Volume Pivot Points
look_forward=20

### Process

In [249]:
reload_modules()

# Step 0: Initialize Labels Dataframe
Labels_Data = Class.Data.Labels.Labels(Features_Data.data)

# Step 1: Categorize Pivot Points
Labels_Data.categorize_pivot_points(look_forward)

# Step 2: Categorize Volume Pivot Points
Labels_Data.categorize_colume_pivot_points(look_forward)

[2025-06-15 17:35:01] INFO - DataFetcher.py - Logger initialized (DataFetcher.py)
[2025-06-15 17:35:01] INFO - Features.py - Logger initialized (Features.py)
[2025-06-15 17:35:01] INFO - Labels.py - Logger initialized (Labels.py)
[2025-06-15 17:35:01] INFO - DataAnalysis.py - Logger initialized (DataAnalysis.py)
[2025-06-15 17:35:01] INFO - Preprocessing.py - Logger initialized (Preprocessing.py)
[2025-06-15 17:35:01] INFO - LSTMModel.py - Logger initialized (LSTMModel.py)
[2025-06-15 17:35:01] INFO - LSTMTrainer.py - Logger initialized (LSTMTrainer.py)
[2025-06-15 17:35:01] INFO - Labels.py - Starting Categorize_Pivot_Points with look_forward=20
[2025-06-15 17:35:01] INFO - Labels.py - Categorize_Pivot_Points completed successfully.
[2025-06-15 17:35:01] INFO - Labels.py - Starting Categorize_Volume_Pivot_Points with look_forward=20


<class 'pandas.core.series.Series'>
<class 'pandas.core.series.Series'>
<class 'pandas.core.series.Series'>
<class 'pandas.core.series.Series'>
<class 'pandas.core.series.Series'>
<class 'pandas.core.series.Series'>
<class 'pandas.core.series.Series'>
<class 'pandas.core.series.Series'>
<class 'pandas.core.series.Series'>
<class 'pandas.core.series.Series'>
<class 'pandas.core.series.Series'>
<class 'pandas.core.series.Series'>
<class 'pandas.core.series.Series'>
<class 'pandas.core.series.Series'>
<class 'pandas.core.series.Series'>
<class 'pandas.core.series.Series'>
<class 'pandas.core.series.Series'>
<class 'pandas.core.series.Series'>
<class 'pandas.core.series.Series'>
<class 'pandas.core.series.Series'>
<class 'pandas.core.series.Series'>
<class 'pandas.core.series.Series'>
<class 'pandas.core.series.Series'>
<class 'pandas.core.series.Series'>
<class 'pandas.core.series.Series'>
<class 'pandas.core.series.Series'>
<class 'pandas.core.series.Series'>
<class 'pandas.core.series.S

[2025-06-15 17:35:01] INFO - Labels.py - Categorize_Volume_Pivot_Points completed successfully.


### Visualization

In [231]:
#-------------------------------------------------------------------------------

print("#----------------------------------------------------------------------")
print("Shape:", Labels_Data.data.shape)
print("#----------------------------------------------------------------------")
print(Labels_Data.data.columns)
print("#----------------------------------------------------------------------")
print(Labels_Data.data.head())
print("#----------------------------------------------------------------------")

#-------------------------------------------------------------------------------

df = Labels_Data.data.copy()[-300:]

# Fréquences des 0 et 1 pour les colonnes cibles
frequency = df[["Low_Below_Pivot", "High_Above_Pivot"]].apply(lambda col: col.value_counts()).fillna(0).astype(int)
print("\nLabel Frequency (0 = no breakout, 1 = breakout):")
print(frequency.head(3))

# Initialiser les couleurs
colors_combined = ['lightgrey'] * len(df)
for i in range(len(df)):
    if df.iloc[i]['High_Above_Pivot'] == 1:
        colors_combined[i] = 'green'
    elif df.iloc[i]['Low_Below_Pivot'] == 1:
        colors_combined[i] = 'red'

# Création du graphique
fig = go.Figure()

# Trace VWAP_5m avec couleur dynamique
fig.add_trace(go.Scatter(
    x=df.index,
    y=df['VWAP_5m'],
    mode='markers+lines',
    marker=dict(color=colors_combined, size=6),
    name='VWAP_5m',
    line=dict(color='lightgrey')
))

# Trace Low_Pivot (ligne continue)
fig.add_trace(go.Scatter(
    x=df.index,
    y=df['Low_Pivot'],
    mode='lines',
    line=dict(color='red', width=1),
    name='Low_Pivot'
))

# Trace High_Pivot (ligne continue)
fig.add_trace(go.Scatter(
    x=df.index,
    y=df['High_Pivot'],
    mode='lines',
    line=dict(color='green', width=1),
    name='High_Pivot'
))

# Mise en page
fig.update_layout(
    title='VWAP_5m with Pivot Breakout Coloring and Pivot Lines',
    xaxis_title='Time',
    yaxis_title='Price',
    template='plotly_dark',
    height=500
)

fig.show()
print("#----------------------------------------------------------------------")

#-------------------------------------------------------------------------------

# Fréquences des 0 et 1 pour les colonnes VWAP
frequency = df[["VWAP_Below_Volume_Low", "VWAP_Above_Volume_High"]].apply(lambda col: col.value_counts()).fillna(0).astype(int)
print("\nLabel Frequency (0 = no breakout, 1 = breakout):")
print(frequency.head(3))

# Initialiser les couleurs
colors_combined = ['lightgrey'] * len(df)
for i in range(len(df)):
    if df.iloc[i]['VWAP_Above_Volume_High'] == 1:
        colors_combined[i] = 'green'
    elif df.iloc[i]['VWAP_Below_Volume_Low'] == 1:
        colors_combined[i] = 'red'

# Création du graphique
fig = go.Figure()

# Trace VWAP_5m avec couleur dynamique
fig.add_trace(go.Scatter(
    x=df.index,
    y=df['VWAP_5m'],
    mode='markers+lines',
    marker=dict(color=colors_combined, size=6),
    name='VWAP_5m',
    line=dict(color='lightgrey')
))

# Trace Volume_Low_Pivot (ligne continue)
fig.add_trace(go.Scatter(
    x=df.index,
    y=df['Volume_Low_Pivot'],
    mode='lines',
    line=dict(color='red', width=1),
    name='Volume_Low_Pivot'
))

# Trace Volume_High_Pivot (ligne continue)
fig.add_trace(go.Scatter(
    x=df.index,
    y=df['Volume_High_Pivot'],
    mode='lines',
    line=dict(color='green', width=1),
    name='Volume_High_Pivot'
))

# Mise en page
fig.update_layout(
    title='VWAP_5m with Volume Pivot Breakout Coloring and Pivot Lines',
    xaxis_title='Time',
    yaxis_title='Price',
    template='plotly_dark',
    height=500
)

fig.show()

#-------------------------------------------------------------------------------

#----------------------------------------------------------------------
Shape: (293, 31)
#----------------------------------------------------------------------
Index(['Open', 'High', 'Low', 'Close', 'Volume', 'bidSize', 'askSize',
       'VWAP_5m', 'London_Open', 'NY_Open', 'HK_Open', 'Dif_Low_Pivot',
       'Dif_High_Pivot', 'Low_Pivot', 'High_Pivot', 'Rolling_VWAP_240min',
       'Volume_Low_Pivot', 'Volume_High_Pivot', 'Dif_Volume_Low_Pivot',
       'Dif_Volume_High_Pivot', 'volume_delta', 'CVD', 'obi', 'price_change',
       'reaction_ratio', 'rolling_std_price', 'rolling_mean_cvd',
       'Low_Below_Pivot', 'High_Above_Pivot', 'VWAP_Below_Volume_Low',
       'VWAP_Above_Volume_High'],
      dtype='object')
#----------------------------------------------------------------------
                         Open      High       Low     Close      Volume  \
time                                                                      
2025-06-11 21:55:00  1.148820  1.148820  1.148820  1.148

#----------------------------------------------------------------------

Label Frequency (0 = no breakout, 1 = breakout):
   VWAP_Below_Volume_Low  VWAP_Above_Volume_High
0                    264                     270
1                     29                      23
